# Example: Forward Validation of a SIM-Derived Portfolio
In this example, we construct a minimum-variance portfolio from Single Index Model parameters and propagate parameter uncertainty into a distribution of forward portfolio risk.

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
>
> * __Construct SIM reward and risk inputs:__ Build the expected-growth vector and covariance matrix from market and asset-level parameters.
> * __Distinguish allocation and validation uncertainty:__ Hold the calibrated portfolio fixed while drawing plausible forward parameter scenarios.
> * __Measure optimization regret:__ Compare the fixed portfolio variance with the scenario-specific minimum achievable variance.

Let's test how much confidence the estimated SIM inputs justify.
___


## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading the required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates `Include.jl` in the notebook's global scope. The file activates the course environment and loads the required packages.

Let's set up the code environment:

The reusable portfolio algorithms in this example are provided by the local [`VLQuantitativeFinancePackage.jl`](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/) package.


In [ ]:
include(joinpath(@__DIR__, "Include.jl"));


For additional information, see the [Julia documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

___


## Task 1: Construct the SIM-Derived Portfolio Inputs
For asset $i$,
$$
g_i=\alpha_i+\beta_i g_m+\varepsilon_i,
$$
so the annualized expected-growth and covariance inputs are
$$
\mu_i=\alpha_i+\beta_i\mu_m,
\qquad
\Sigma_{ij}=\beta_i\beta_j\sigma_m^2+\mathbf 1_{i=j}\sigma_{\varepsilon,i}^2.
$$


In [ ]:
Random.seed!(5660);
tickers = ["ALFA", "BRAV", "CHAR", "DELT", "ECHO", "FOXT"];
μm = 0.08;
σm = 0.18;
α = [0.010, 0.015, 0.006, 0.012, 0.020, 0.009];
β = [0.70, 0.90, 1.10, 1.25, 1.45, 0.80];
σε = [0.11, 0.13, 0.16, 0.18, 0.23, 0.12];

μ̂, Σ̂ = sim_portfolio_inputs(α, β, σε, μm, σm);
ŵ = minimum_variance_weights(Σ̂);

calibration = DataFrame(ticker=tickers, alpha=α, beta=β,
    residual_volatility=σε, expected_growth=μ̂, portfolio_weight=ŵ);
pretty_table(calibration; table_format=TextTableFormat(borders=text_table_borders__simple))


## Task 2: Draw Forward Parameter Scenarios
The fitted parameters are estimates, not constants. We model plausible estimation error with independent draws around each calibrated value. This simplified experiment is not a substitute for a full bootstrap, but it makes the propagation step explicit.

For every scenario we evaluate:

- the variance of the fixed calibrated allocation $\widehat{\mathbf w}$;
- the scenario-specific minimum variance;
- variance regret, defined as their difference.


In [ ]:
number_of_scenarios = 2500;
αse = fill(0.006, length(α));
βse = [0.08, 0.09, 0.11, 0.12, 0.14, 0.09];
log_σε_se = 0.12;

fixed_variance = Vector{Float64}(undef, number_of_scenarios);
optimal_variance = similar(fixed_variance);
fixed_growth = similar(fixed_variance);
weight_distance = similar(fixed_variance);

for s in 1:number_of_scenarios
    αs = α .+ αse.*randn(length(α));
    βs = β .+ βse.*randn(length(β));
    σεs = σε .* exp.(log_σε_se.*randn(length(σε)));
    μms = μm + 0.025*randn();
    σms = max(0.08, σm*exp(0.15*randn()));
    μs, Σs = sim_portfolio_inputs(αs, βs, σεs, μms, σms);
    ws = minimum_variance_weights(Σs);
    fixed_variance[s] = dot(ŵ, Σs*ŵ);
    optimal_variance[s] = dot(ws, Σs*ws);
    fixed_growth[s] = dot(ŵ, μs);
    weight_distance[s] = 0.5*sum(abs.(ws .- ŵ));
end

variance_regret = fixed_variance .- optimal_variance;


## Task 3: Build the Forward-Validation Scorecard
The scorecard reports the calibrated values and selected quantiles from the scenario distribution. Variance regret is nonnegative up to numerical error because each scenario-specific optimizer is allowed to use the scenario inputs that the fixed portfolio does not know.


In [ ]:
scorecard = DataFrame(
    metric=["Annual expected growth", "Annual volatility", "Variance regret", "Weight distance"],
    calibrated=[dot(ŵ, μ̂), sqrt(dot(ŵ, Σ̂*ŵ)), 0.0, 0.0],
    q05=[quantile(fixed_growth,0.05), quantile(sqrt.(fixed_variance),0.05),
        quantile(variance_regret,0.05), quantile(weight_distance,0.05)],
    median=[median(fixed_growth), median(sqrt.(fixed_variance)),
        median(variance_regret), median(weight_distance)],
    q95=[quantile(fixed_growth,0.95), quantile(sqrt.(fixed_variance),0.95),
        quantile(variance_regret,0.95), quantile(weight_distance,0.95)],
);
pretty_table(scorecard; table_format=TextTableFormat(borders=text_table_borders__simple))


In [ ]:
histogram(sqrt.(fixed_variance), bins=35, normalize=:pdf,
    c=:navy, alpha=0.65, label="Forward volatility",
    xlabel="Annualized portfolio volatility", ylabel="Density")
vline!([sqrt(dot(ŵ, Σ̂*ŵ))], c=:red, lw=2, ls=:dash,
    label="Calibration estimate")


## Summary
This example propagated uncertainty in the SIM inputs into the portfolio quantities used for decision-making.

> __Key Takeaways:__
>
> * __Estimated parameters create a distribution of plausible portfolios:__ A single calibrated covariance matrix hides uncertainty in market exposure and residual risk.
> * __A fixed allocation can be evaluated without pretending to know the future optimum:__ Forward scenarios measure how the deployed weights behave under alternative inputs.
> * __Optimization regret provides a diagnostic:__ It measures the variance penalty associated with using calibration-period weights when the forward covariance changes.

This forward-validation step prepares the portfolio for the return, correlation, and transaction-cost stress tests in L7b.
___

## Disclaimer and Risks
This material is for educational purposes only and does not constitute investment advice. The parameter-error model is simplified and should not be interpreted as a calibrated forecast distribution.
